# Day 12 — Solution: Constrained Minimum Variance

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy.optimize import minimize
DATA_SOURCE = os.environ.get("QRC_DATA", "real")
from qrc.data import get_prices
from qrc.synth import synthetic_prices
from qrc.universe import load_universe

if DATA_SOURCE == "real":
    tickers = load_universe("core_etfs")[:8]
    px = get_prices(tickers, start="2015-01-01")
else:
    tickers = [f"S{i}" for i in range(8)]
    px = synthetic_prices(n_days=2500, n_assets=8, seed=71, corr=0.3)
    px.columns = tickers
rets = px.pct_change().dropna()
Sigma = rets.cov().values * 252

## E1 — the solver

In [ ]:
def min_var(Sigma, long_only=True, gross_cap=None):
    n = Sigma.shape[0]
    cons = [{"type": "eq", "fun": lambda w: w.sum() - 1.0}]
    if gross_cap is not None:
        cons.append({"type": "ineq", "fun": lambda w: gross_cap - np.abs(w).sum()})
    bounds = [(0.0, 1.0)] * n if long_only else [(-2.0, 2.0)] * n
    res = minimize(lambda w: w @ Sigma @ w, x0=np.full(n, 1 / n),
                   constraints=cons, bounds=bounds, method="SLSQP")
    assert res.success, res.message
    w = res.x
    assert np.isclose(w.sum(), 1.0, atol=1e-6)
    if long_only:
        assert (w >= -1e-9).all()
    return w

w_lo, w_unc = min_var(Sigma, True), min_var(Sigma, False)

## E2 — three regimes

In [ ]:
def report(w, label):
    vol = np.sqrt(w @ Sigma @ w)
    top = sorted(zip(tickers, w), key=lambda t: -abs(t[1]))[:3]
    print(f"{label}: vol {vol:.2%} | sum {w.sum():.4f} | min {w.min():.3f} "
          f"| max {w.max():.3f} | top {[(t, round(x,3)) for t,x in top]}")

report(w_lo, "long-only ")
report(w_unc, "unconstr. ")

Long-only piles into the lowest-vol assets (typically XLU/TLT side of the
universe) and pins several weights at exactly 0 (binding bounds).
Unconstrained wants large offsetting longs/shorts among correlated assets —
"optimal" only for an investor with free infinite borrowing/shorting at no
cost: operationally fictional.

## E3 — stability, honestly measured

In [ ]:
half = len(rets) // 2
A, B = rets.iloc[:half], rets.iloc[half:]
SigA, SigB = A.cov().values * 252, B.cov().values * 252
n = len(tickers)

w_A = min_var(SigA, True)                                   # estimated on A
w_eq = np.full(n, 1 / n)
w_cheat = min_var(SigB, True)                               # IN-SAMPLE on B

vol_A_on_B = np.sqrt(w_A @ SigB @ w_A)
vol_eq_on_B = np.sqrt(w_eq @ SigB @ w_eq)
vol_cheat = np.sqrt(w_cheat @ SigB @ w_cheat)

print(f"w_A (out-of-sample): {vol_A_on_B:.2%}")
print(f"1/N:                {vol_eq_on_B:.2%}")
print(f"cheat (in-sample B): {vol_cheat:.2%}   <- never report this as performance")
print(f"value of perfect info: {vol_eq_on_B - vol_cheat:+.2%}")
print(f"cost of estimation error: {vol_eq_on_B - vol_A_on_B:+.2%}")

Typical result: the perfect-info gap is a few vol points; estimation error
eats most or all of it — w_A often *loses* to 1/N out of sample. That is
DeMiguel–Garlappi–Uppal's (2009) finding, felt locally. (One split proves
nothing in general — E5.)

## E4 — the leverage cap

In [ ]:
w_cap = min_var(Sigma, long_only=False, gross_cap=1.6)
report(w_cap, "gross<=1.6 ")

The capped solution sits between the extremes: modest shorts in the most
redundant assets, no exploding offsetting book. The cap buys *operational
reality*: bounded borrow needs, bounded turnover toward the optimum, and
robustness to Σ's estimation error (less freedom = less error
amplification). This is why real mandates are full of caps.

## E5 — why one split settles nothing

1. **Multiplicity**: one split is one draw; the volatility of the A-vs-1/N
   comparison across splits is large relative to its mean.
2. **Regime dependence**: half B may be a calm or stormy regime — the
   answer flips by construction of the split.
3. **Selection**: choosing the split/period after peeking at results is
   orientation-day-5's champion, wearing a portfolio hat.
Module 13's answer: walk-forward evaluation across many splits, reported as
a distribution.